In [ ]:
import numpy as np
import cv2
import mediapipe as mp


import time
import math as m

In [ ]:
mp_drawing = mp.solutions.drawing_utils
hand = mp.solutions.hands
mp_hands = mp.solutions.hands.Hands()

cap=cv2.VideoCapture(0)

pTime=0
temp = None
tan_theta = None
angle = None

previous_angle = 0.0

def calculate_angle(A, B, C):
    """Calculate the angle at point B formed by angle ABC."""
    global previous_angle

    # Vectors BA and BC
    BAx, BAy = A[0] - B[0], A[1] - B[1]
    BCx, BCy = C[0] - B[0], C[1] - B[1]

    dot_prod = BAx * BCx + BAy * BCy
    mag_BA = m.sqrt(BAx**2 + BAy**2)
    mag_BC = m.sqrt(BCx**2 + BCy**2)

    if mag_BA == 0 or mag_BC == 0:
        return previous_angle

    cos_theta = dot_prod / (mag_BA * mag_BC)
    cos_theta = max(-1.0, min(1.0, cos_theta))

    angle_radians = m.acos(cos_theta)
    angle_degrees = m.degrees(angle_radians)

    previous_angle = round(angle_degrees, 2)
    return previous_angle

while cap.isOpened():
    ret,image=cap.read()
    if ret==False :
        break
    rows, cols, _ = image.shape

    image = cv2.cvtColor(cv2.flip(image, 1), cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = mp_hands.process(image)
    image.flags.writeable = True

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            x = [ int(landmark.x * cols) for landmark in hand_landmarks.landmark]
            y = [ int(landmark.y * rows) for landmark in hand_landmarks.landmark]

            mp_drawing.draw_landmarks( image, hand_landmarks,hand.HAND_CONNECTIONS)
            cv2.circle(image,(x[8],y[8]),8,(255,255,0),2)
            cv2.circle(image,(x[5],y[5]),8,(255,255,0),2)
            """distance = abs(y[5]-y[8])
            if distance > 40 :
                temp = "up"
            else:
                temp = 'down'"""

            """if y[8] < y[5] :
                temp = "up"
            else:
                temp = 'down' """

            angle = calculate_angle(A=(x[8],y[8]),B=(x[5],y[5]),C=(x[0],y[0]))
            if angle > 50 :
                temp = "up"
            else:
                temp = 'down'
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    cTime=time.time()
    fps=1/(cTime-pTime)
    pTime=cTime
    cv2.putText(image, f'{fps:0.0f} , {angle} : {temp}', (10,50), cv2.FONT_HERSHEY_SIMPLEX,1.2,(255,0,0), 3)

    cv2.imshow('MediaPipe Hands', image)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
cap.release()
cv2.destroyAllWindows()